# 1. Information about the submission

## 1.1 Name and number of the assignment

Hallucination Detection in Tool Calling

## 1.2 Student name

Erik Shaikhiev

Tishchenko Margarita

Artemii Rubtcov

Roman Branovets

Andrej Mymrin

## 1.3 Codalab user ID / nickname / username

Not applicable for this local reproducibility notebook.

## 1.4 Additional comments

The notebook is written to be reproducible both inside the local repository and in a clean Colab-like environment. It uses the public GitHub repository `Eroouu/transformers_project.git` and the prepared ToolACE-derived datasets when they are available.

# 2. Technical Report

## 2.1 Methodology

The assignment asks us to detect span-level hallucinations in tool-calling dialogues. Following the provided task description, each example is represented in a RAGTruth-style schema: `query` is the user question, `context` is the tool output, `output` is the final model answer, and `hallucination_labels` contains character-level spans that mark unsupported text. The project builds on ToolACE dialogues and creates three corrupted subsets: contradiction with the tool output, overgeneration beyond the tool output, and missing-tool suggestions that require unavailable tools. Clean examples are kept as negative cases.

The repository implements the full pipeline in Python. Dataset construction and validation live in `data/`, while training and evaluation live in `src/`. The main improved model is a LettuceDetect-compatible token classifier fine-tuned on the generated span labels. During preprocessing, the tool output and user query are formatted as context/question, the final answer is tokenized as the second sequence, and only answer tokens receive binary labels: `supported` or `hallucination`. To handle class imbalance, the trainer can use inverse-frequency class weights.

For a reproducible run, the notebook first locates or clones `https://github.com/Eroouu/transformers_project.git`, installs the project dependencies, validates the final dataset split, trains the token classifier on `final_dataset_train`, and evaluates on `final_dataset_test`. The default configuration uses a short `max_steps` smoke training run so that the notebook can execute on limited hardware; setting `QUICK_RUN = False` switches to the full training schedule. Metrics are span-level Precision, Recall, and F1 computed by the repository evaluator through overlap between predicted and gold hallucination spans.

## 2.1 Dataset preparation
We tried different ways to add hallucinations. First attempt to create hallucunations via python script was unsucsessful -
### 2.1.1 Qwen model assisted generation
 In `generate_hallucinations_qwen2.py`, hallucinations were added by modifying correct tool-grounded responses and creating three corrupted versions: contradiction, overgeneration, and missing_tool. Contradiction was created by replacing a supported fact with an incorrect one, overgeneration by adding a plausible but unsupported detail, and missing_tool by adding a sentence where the assistant claimed it could perform an action not available in the given context.

The script used both a local Qwen model and rule-based fallback templates to generate these corruptions. The share of examples generated by the LLM was controlled by the `llm_fraction` parameter, while the remaining examples were produced algorithmically. Each hallucinated fragment was then labeled with its exact position, type, and generation strategy, which made the dataset suitable for further analysis and evaluation.

### 2.1.2 GPT-4o-mini assisted generation

We also experimented with GPT-4o-mini to improve the quality and diversity of generated hallucinations. Instead of relying only on local rules or the Qwen model, we used GPT-4o-mini to produce more natural corrupted responses while keeping the same three hallucination types: contradiction, overgeneration, and missing_tool.

For each example, GPT-4o-mini received the user query, tool context, and original grounded answer. It was asked to minimally edit the answer by inserting or replacing only the hallucinated fragment, while preserving the rest of the response as much as possible. This helped create corruptions that looked more fluent and realistic than simple template-based edits.

After generation, we still applied automatic validation. The hallucinated text had to appear exactly in the final output so that its character offsets could be computed reliably. If the model output did not satisfy this requirement, the example was rejected or replaced by a rule-based fallback. This kept the final dataset compatible with span-level evaluation.

We combined resultes of both models (Qwen & GPT-4o-mini) into final dataset that was used in all further experiments.





## 2.2 Ensamble method: first tried approach

 The first approach involved constructing a weighted voting classifier integrating the three baseline models. The output logs from the baselines were aggregated using distinct weighting coefficients: 0.25 for the tool_overlap and lettucedetect baselines, and 0.5 for the lookback_lens baseline. These coefficients were selected based on their optimal performance metrics on the training dataset. The resultant aggregated logs served as the basis for determining the final binary classification outcomes.

In our experiments, the ensemble improved over some individual baselines, especially compared with LettuceDetect on the final split, but it remained limited by the false positives inherited from the high-recall components.

## 2.3 Final method Description
## Idea
We train a context-aware span-level hallucination detector for tool-calling dialogues. Given a user query, a tool output, and the assistant’s final answer, the model predicts which parts of the assistant answer are hallucinated.

The task is formulated as token-level binary classification over the assistant answer. The model receives the full input:
$
(q, c, y)
$

where:

- $q$ is the user query,
- $c$ is the tool output or context,
- $y$ is the assistant answer.

Only tokens from the assistant answer are supervised. Tokens belonging to the user query and tool output are masked out during training.

Let the answer be tokenized as:

$$
y = (t_1, t_2, \dots, t_n)
$$

For each answer token $t_i$, we assign a binary label:

$$
z_i =
\begin{cases}
1, & \text{if } t_i \text{ overlaps with a gold hallucination span} \\
0, & \text{otherwise}
\end{cases}
$$

The model is a transformer-based token classifier that predicts:

$$
p_i = P(z_i = 1 \mid q, c, y)
$$

where $p_i$ is the predicted probability that answer token $t_i$ belongs to a hallucinated span.

The default base model is:
*KRLabsOrg/lettucedect-base-modernbert-en-v1*

## Training Objective

Hallucination span detection is highly imbalanced because most answer tokens are supported, while hallucinated tokens are relatively rare. To address this, we use **class-weighted focal loss**.

For each supervised answer token $i$, let:

$$
p_i = \text{predicted probability assigned to the true class}
$$

$$
w_{z_i} = \text{class weight for label } z_i
$$

$$
\gamma = \text{focal loss parameter}
$$

The token-level loss is:

$$
L_i = -w_{z_i}(1 - p_i)^\gamma \log(p_i)
$$

The full training loss is averaged only over supervised answer tokens:

$$
L = \frac{1}{N}\sum_{i=1}^{N} L_i
$$

Tokens from the query and tool output are assigned the label $-100$, so they are ignored by the loss function.

Class weights are computed from the training data as:

$$
w_k = \frac{N}{2N_k}
$$

where:

$$
N = \text{total number of labeled answer tokens}
$$

$$
N_k = \text{number of answer tokens in class } k
$$

$$
k \in \{\text{supported}, \text{hallucination}\}
$$

This increases the contribution of rare hallucination tokens during training.

##Why This Improves Over Baselines

A simple tool_overlap baseline flags answer tokens that do not appear in the tool output. This can achieve high recall, but it often has low precision because many correct answer tokens are not copied literally from the tool output.

LookBackLens uses attention-based lookback features from a causal language model. These features can provide useful signals, but attention ratios are noisy and require a separate classifier.

LettuceDetect is the strongest baseline because it is already a transformer-based hallucination detector. However, the off-the-shelf model is general-purpose. Our method improves on it by fine-tuning directly on ToolACE-style corruption types and by tuning span decoding on the validation set.

The main advantages of the proposed method are:

Fine-tuning on the target task distribution,

*   Answer-only supervision,
*   Class-weighted focal loss for rare hallucination tokens,
*  Leakage-safe validation splitting,
*  Validation-tuned probability threshold,
*  Span-level post-processing,
* Prompt truncation that preserves answer tokens.


Overall, the method is designed to optimize the actual leaderboard objective: span-level hallucination detection F1.

Model and all training and evaluation are in `src/finetuned_span_model.py` file.  

## 2.2 Discussion of results

Contradiction
| model                  |  tp |  fp |  fn | precision |   recall |       f1 | n_examples |
| ---------------------- | --: | --: | --: | --------: | -------: | -------: | ---------: |
| ensemble_weighted_vote | 451 | 583 |  48 |  0.436170 | 0.903808 | 0.588389 |        486 |
| lettucedetect          | 376 | 823 | 121 |  0.313600 | 0.756500 | 0.443400 |        486 |
| lookback_lens          | 452 | 553 |  36 |  0.449800 | 0.926200 | 0.605500 |        486 |
| Fine-Tuned Span Model      | 403 |  42 |  86 |  0.905600 | 0.824100 | 0.863000 |        486 |

Overgeneration
| model                  |  tp |  fp | fn | precision |   recall |       f1 | n_examples |
| ---------------------- | --: | --: | -: | --------: | -------: | -------: | ---------: |
| ensemble_weighted_vote | 586 | 636 | 35 |  0.479542 | 0.943639 | 0.635920 |        486 |
| lettucedetect          | 451 | 837 | 72 |  0.350200 | 0.862300 | 0.498100 |        486 |
| lookback_lens          | 487 | 615 |  1 |  0.441900 | 0.998000 | 0.612600 |        486 |
| Fine-Tuned Span Model      | 489 |  10 |  2 |  0.980000 | 0.995900 | 0.987900 |        486 |

Missing tool
| model                  |  tp |  fp |  fn | precision |   recall |       f1 | n_examples |
| ---------------------- | --: | --: | --: | --------: | -------: | -------: | ---------: |
| ensemble_weighted_vote | 435 | 659 | 116 |  0.397623 | 0.789474 | 0.528875 |        486 |
| lettucedetect          | 289 | 900 | 281 |  0.243100 | 0.507000 | 0.328600 |        486 |
| lookback_lens          | 486 | 621 |   1 |  0.439000 | 0.997900 | 0.609800 |        486 |
| Fine-Tuned Span Model      | 484 |   7 |   2 |  0.985700 | 0.995900 | 0.990800 |        486 |

Overall
| model             |   tp |    fp |  fn | precision |   recall |       f1 | n_examples |
| ----------------- | ---: | ----: | --: | --------: | -------: | -------: | ---------: |
| tool_overlap      | 2099 | 13582 |  18 |  0.133900 | 0.991500 | 0.235900 |       1944 |
| lettucedetect     | 1116 |  3457 | 474 |  0.244000 | 0.701900 | 0.362200 |       1944 |
| lookback_lens     | 1425 |  2623 |  38 |  0.352000 | 0.974000 | 0.517100 |       1944 |
| Fine-Tuned Span Model | 1376 |   113 |  90 |  0.924100 | 0.938600 | 0.931300 |       1944 |

The **Fine-Tuned Span Model** clearly outperforms all baselines across all corruption types. Overall, it achieves an F1 score of **0.9313**, compared with **0.5171** for the strongest baseline, LookBackLens.

The main improvement comes from much higher precision. Baseline methods achieve high recall but produce many false positives. For example, LookBackLens has **2623** false positives overall, and the `tool_overlap` baseline has **13582**. In contrast, the Fine-Tuned Span Model produces only **113** false positives while still maintaining high recall.

### Contradiction

On contradiction examples, the Fine-Tuned Span Model achieves the best F1 score:

$$
F1 = 0.8630
$$

It has much higher precision than the baselines:

$$
P = 0.9056
$$

However, its recall is slightly lower than LookBackLens and the ensemble. This suggests that contradiction cases are more semantically difficult: the model is very accurate when it predicts a span, but it misses some subtle contradictions.

### Overgeneration

The model performs extremely well on overgeneration examples:

$$
F1 = 0.9879
$$

It achieves both high precision and high recall:

$$
P = 0.9800, \quad R = 0.9959
$$

Compared with the baselines, it dramatically reduces false positives. LookBackLens produces **615** false positives, while the Fine-Tuned Span Model produces only **10**.

### Missing Tool

The strongest result is on missing-tool examples:

$$
F1 = 0.9908
$$

The model reaches:

$$
P = 0.9857, \quad R = 0.9959
$$

Again, the improvement mainly comes from precision. LookBackLens has similar recall, but it produces **621** false positives. The Fine-Tuned Span Model reduces this to only **7**.

### Overall Interpretation

The baselines mostly behave as high-recall but low-precision detectors. They find many hallucinated spans, but they also incorrectly flag many supported tokens. The Fine-Tuned Span Model gives a much better precision-recall balance:

$$
P = 0.9241, \quad R = 0.9386, \quad F1 = 0.9313
$$

These results show that task-specific fine-tuning, answer-only supervision, focal loss, and validation-tuned span decoding are effective for span-level hallucination detection.

# 3. Code

## 3.1 Requirements

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Eroouu/transformers_project.git"
REPO_DIR = Path("transformers_project")

cwd = Path.cwd()
if (cwd / "src" / "train_lettucedetect.py").exists() and (cwd / "requirements.txt").exists():
    project_dir = cwd
else:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    project_dir = REPO_DIR.resolve()
    os.chdir(project_dir)

print(f"Project directory: {Path.cwd()}")
print(f"Repository URL: {REPO_URL}")

INSTALL_DEPS = True
if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


Project directory: /content/transformers_project
Repository URL: https://github.com/Eroouu/transformers_project.git


In [ ]:
!pip install --upgrade scikit-learn  #colab uses old version that causes problems when running baselines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 93.1 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


## 3.2 Download the data

In [ ]:
from pathlib import Path

DATASET_DIR = Path("final_dataset")
TRAIN_DIR = Path("final_dataset_train")
TEST_DIR = Path("final_dataset_test")
DATASET_FILES = ["clean.jsonl", "contradiction.jsonl", "overgeneration.jsonl", "missing_tool.jsonl"]

for dataset_dir in [DATASET_DIR, TRAIN_DIR, TEST_DIR]:
    missing = [name for name in DATASET_FILES if not (dataset_dir / name).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing files in {dataset_dir}: {missing}. "
            "Generate the dataset with scripts from data/ or pull the prepared artifacts."
        )

print("Found prepared datasets:")
for dataset_dir in [DATASET_DIR, TRAIN_DIR, TEST_DIR]:
    print(f"- {dataset_dir.resolve()}")


Found prepared datasets:
- /content/transformers_project/final_dataset
- /content/transformers_project/final_dataset_train
- /content/transformers_project/final_dataset_test


In [ ]:
import json
from pprint import pprint

sample_path = TEST_DIR / "contradiction.jsonl"
with sample_path.open("r", encoding="utf-8") as f:
    sample = json.loads(next(f))

pprint({
    "query": sample.get("query", "")[:300],
    "context": sample.get("context", "")[:300],
    "output": sample.get("output", "")[:300],
    "hallucination_labels": sample.get("hallucination_labels", []),
    "corruption_type": sample.get("corruption_type"),
})


{'context': '[{"name": "Quotes by Keywords", "results": {"quotes": [{"text": '
            '"The only way to achieve the impossible is to believe it is '
            'possible.", "author": "Charles Kingsleigh"}, {"text": "Don\'t '
            'watch the clock; do what it does. Keep going.", "author": "Sam '
            'Levenson"}, {"text": "Success is not the key to happ',
 'corruption_type': 'contradiction',
 'hallucination_labels': [{'end': 127,
                           'label': 'hallucination',
                           'start': 119,
                           'text': 'John Doe',
                           'type': 'contradiction'}],
 'output': 'Here are some inspiration quotes for you:\n'
           '\n'
           '1. "The only way to achieve the impossible is to believe it is '
           'possible." - John Doe\n'
           '2. "Don\'t watch the clock; do what it does. Keep going." - Sam '
           'Levenson\n'
           '3. "Success is not the key to happiness. Happiness 

## 3.3 Preprocessing

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd


def summarize_dataset(dataset_dir: Path) -> pd.DataFrame:
    rows = []
    for name in DATASET_FILES:
        path = dataset_dir / name
        examples = 0
        examples_with_labels = 0
        spans = 0
        span_chars = 0
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                labels = item.get("hallucination_labels", [])
                examples += 1
                examples_with_labels += int(bool(labels))
                spans += len(labels)
                span_chars += sum(int(label["end"]) - int(label["start"]) for label in labels)
        rows.append({
            "split": dataset_dir.name,
            "file": name,
            "examples": examples,
            "examples_with_labels": examples_with_labels,
            "gold_spans": spans,
            "avg_span_chars": round(span_chars / spans, 2) if spans else 0.0,
        })
    return pd.DataFrame(rows)

stats = pd.concat(
    [summarize_dataset(DATASET_DIR), summarize_dataset(TRAIN_DIR), summarize_dataset(TEST_DIR)],
    ignore_index=True,
)
display(stats)

subprocess.run([sys.executable, "data/validate_corrupted_datasets.py", str(DATASET_DIR)], check=True)


,split,file,examples,examples_with_labels,gold_spans,avg_span_chars
0,final_dataset,clean.jsonl,2431,0,0,0.00
1,final_dataset,contradiction.jsonl,2431,2431,2431,16.28
2,final_dataset,overgeneration.jsonl,2431,2431,2431,66.61
3,final_dataset,missing_tool.jsonl,2431,2431,2431,63.06
4,final_dataset_train,clean.jsonl,1945,0,0,0.00
5,final_dataset_train,contradiction.jsonl,1945,1945,1945,16.08
6,final_dataset_train,overgeneration.jsonl,1945,1945,1945,66.75
7,final_dataset_train,missing_tool.jsonl,1945,1945,1945,62.94
8,final_dataset_test,clean.jsonl,486,0,0,0.00
9,final_dataset_test,contradiction.jsonl,486,486,486,17.11


CompletedProcess(args=['/usr/bin/python3', 'data/validate_corrupted_datasets.py', 'final_dataset'], returncode=0)

## 3.4 Experiments

### 3.4.1 Tool overlap baseline

In [ ]:
!python src/eval_baselines.py --dataset final_dataset_test --method tool_overlap

Loaded 1944 examples from final_dataset_test (clean.jsonl=486, contradiction.jsonl=486, overgeneration.jsonl=486, missing_tool.jsonl=486)
Building predictor: method=tool_overlap, lettuce_model=KRLabsOrg/lettucedect-base-modernbert-en-v1, lookback_classifier=models/lookback_lens, device=None
Predictor is ready. Starting evaluation...
Evaluating tool_overlap: 100% 1944/1944 [00:00<00:00, 9432.47example/s] 
Method=tool_overlap  TP=2099 FP=13582 FN=18 P=0.1339 R=0.9915 F1=0.2359


###3.4.2 LettuceDetect baseline

In [ ]:
!python src/eval_baselines.py --dataset final_dataset_test --method lettucedetect --device cuda

Loaded 1944 examples from final_dataset_test (clean.jsonl=486, contradiction.jsonl=486, overgeneration.jsonl=486, missing_tool.jsonl=486)
Building predictor: method=lettucedetect, lettuce_model=KRLabsOrg/lettucedect-base-modernbert-en-v1, lookback_classifier=models/lookback_lens, device=cuda
model.safetensors: 100% 598M/598M [00:04<00:00, 129MB/s]
Loading weights: 100% 138/138 [00:00<00:00, 1352.06it/s, Materializing param=model.layers.21.mlp_norm.weight]
Predictor is ready. Starting evaluation...
Evaluating lettucedetect:   0% 0/1944 [00:00<?, ?example/s]W0525 23:15:37.021000 18149 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode
Evaluating lettucedetect: 100% 1944/1944 [01:32<00:00, 21.03example/s]
Method=lettucedetect  TP=1116 FP=3457 FN=474 P=0.2440 R=0.7019 F1=0.3622


In [ ]:
!python src/eval_baselines.py --dataset final_dataset_test/contradiction.jsonl --method lettucedetect --device cuda


Loaded 486 examples from final_dataset_test/contradiction.jsonl
Building predictor: method=lettucedetect, lettuce_model=KRLabsOrg/lettucedect-base-modernbert-en-v1, lookback_classifier=models/lookback_lens, device=cuda
Loading weights: 100% 138/138 [00:00<00:00, 2514.72it/s, Materializing param=model.layers.21.mlp_norm.weight]
Predictor is ready. Starting evaluation...
Evaluating lettucedetect: 100% 486/486 [00:26<00:00, 18.62example/s]
Method=lettucedetect  TP=376 FP=823 FN=121 P=0.3136 R=0.7565 F1=0.4434


In [ ]:
!python src/eval_baselines.py --dataset final_dataset_test/overgeneration.jsonl --method lettucedetect --device cuda


Loaded 486 examples from final_dataset_test/overgeneration.jsonl
Building predictor: method=lettucedetect, lettuce_model=KRLabsOrg/lettucedect-base-modernbert-en-v1, lookback_classifier=models/lookback_lens, device=cuda
Loading weights: 100% 138/138 [00:00<00:00, 1452.30it/s, Materializing param=model.layers.21.mlp_norm.weight]
Predictor is ready. Starting evaluation...
Evaluating lettucedetect: 100% 486/486 [00:26<00:00, 18.27example/s]
Method=lettucedetect  TP=451 FP=837 FN=72 P=0.3502 R=0.8623 F1=0.4981


In [ ]:
!python src/eval_baselines.py --dataset final_dataset_test/missing_tool.jsonl --method lettucedetect --device cuda


Loaded 486 examples from final_dataset_test/missing_tool.jsonl
Building predictor: method=lettucedetect, lettuce_model=KRLabsOrg/lettucedect-base-modernbert-en-v1, lookback_classifier=models/lookback_lens, device=cuda
Loading weights: 100% 138/138 [00:00<00:00, 1545.86it/s, Materializing param=model.layers.21.mlp_norm.weight]
Predictor is ready. Starting evaluation...
Evaluating lettucedetect: 100% 486/486 [00:26<00:00, 18.09example/s]
Method=lettucedetect  TP=289 FP=900 FN=281 P=0.2431 R=0.5070 F1=0.3286


In [ ]:
!python src/eval_baselines.py --dataset final_dataset_test/clean.jsonl --method lettucedetect --device cuda

Loaded 486 examples from final_dataset_test/clean.jsonl
Building predictor: method=lettucedetect, lettuce_model=KRLabsOrg/lettucedect-base-modernbert-en-v1, lookback_classifier=models/lookback_lens, device=cuda
Loading weights: 100% 138/138 [00:00<00:00, 1178.83it/s, Materializing param=model.layers.21.mlp_norm.weight]
Predictor is ready. Starting evaluation...
Evaluating lettucedetect: 100% 486/486 [00:25<00:00, 18.94example/s]
Method=lettucedetect  TP=0 FP=897 FN=0 P=0.0000 R=0.0000 F1=0.0000


###3.4.3 LookbackLens baseline

In [ ]:
!python src/train_lookback_lens.py --dataset final_dataset_train --output_dir models\lookback_lens_final --device cuda

Loaded 7780 examples from 4 file(s).
Backbone model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Sliding window: 8
config.json: 100% 608/608 [00:00<00:00, 3.09MB/s]
tokenizer_config.json: 1.29kB [00:00, 2.30MB/s]
tokenizer.json: 1.84MB [00:00, 72.0MB/s]
tokenizer.model: 100% 500k/500k [00:00<00:00, 602kB/s] 
special_tokens_map.json: 100% 551/551 [00:00<00:00, 3.00MB/s]
model.safetensors: 100% 2.20G/2.20G [00:12<00:00, 183MB/s]
Loading weights: 100% 201/201 [00:01<00:00, 106.54it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 124/124 [00:00<00:00, 450kB/s]
Extracting lookback-ratio features and training classifier...
Training LookBack Lens: 100% 7780/7780 [51:02<00:00,  2.54example/s]
^C


In [ ]:
!python src/eval_baselines.py --dataset final_dataset_test --method lookback_lens --lookback_classifier models/lookback_lens --device cuda

Loaded 1944 examples from final_dataset_test (clean.jsonl=486, contradiction.jsonl=486, overgeneration.jsonl=486, missing_tool.jsonl=486)
Building predictor: method=lookback_lens, lettuce_model=KRLabsOrg/lettucedect-base-modernbert-en-v1, lookback_classifier=models/lookback_lens, device=cuda
config.json: 100% 608/608 [00:00<00:00, 1.85MB/s]
tokenizer_config.json: 1.29kB [00:00, 2.94MB/s]
tokenizer.json: 1.84MB [00:00, 29.3MB/s]
tokenizer.model: 100% 500k/500k [00:00<00:00, 794kB/s] 
special_tokens_map.json: 100% 551/551 [00:00<00:00, 2.42MB/s]
model.safetensors: 100% 2.20G/2.20G [00:16<00:00, 136MB/s]
Loading weights: 100% 201/201 [00:01<00:00, 106.28it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 124/124 [00:00<00:00, 613kB/s]
Predictor is ready. Starting evaluation...
Evaluating lookback_lens: 100% 1944/1944 [13:49<00:00,  2.34example/s]
Method=lookback_lens  TP=1425 FP=2623 FN=38 P=0.3520 R=0.9740 F1=0.5171
Saved evaluation results to models/lookback_lens/e

In [ ]:
!python src/eval_baselines.py --dataset final_dataset_test/contradiction.jsonl --method lookback_lens --lookback_classifier models/lookback_lens --device cuda


Loaded 486 examples from final_dataset_test/contradiction.jsonl
Building predictor: method=lookback_lens, lettuce_model=KRLabsOrg/lettucedect-base-modernbert-en-v1, lookback_classifier=models/lookback_lens, device=cuda
Loading weights: 100% 201/201 [00:01<00:00, 119.61it/s, Materializing param=model.norm.weight]
Predictor is ready. Starting evaluation...
Evaluating lookback_lens: 100% 486/486 [03:16<00:00,  2.47example/s]
Method=lookback_lens  TP=452 FP=553 FN=36 P=0.4498 R=0.9262 F1=0.6055
Saved evaluation results to models/lookback_lens/eval_results.json


In [ ]:
!python src/eval_baselines.py --dataset final_dataset_test/overgeneration.jsonl --method lookback_lens --lookback_classifier models/lookback_lens --device cuda


Loaded 486 examples from final_dataset_test/overgeneration.jsonl
Building predictor: method=lookback_lens, lettuce_model=KRLabsOrg/lettucedect-base-modernbert-en-v1, lookback_classifier=models/lookback_lens, device=cuda
Loading weights: 100% 201/201 [00:02<00:00, 98.51it/s, Materializing param=model.norm.weight] 
Predictor is ready. Starting evaluation...
Evaluating lookback_lens: 100% 486/486 [03:36<00:00,  2.25example/s]
Method=lookback_lens  TP=487 FP=615 FN=1 P=0.4419 R=0.9980 F1=0.6126
Saved evaluation results to models/lookback_lens/eval_results.json


In [ ]:
!python src/eval_baselines.py --dataset final_dataset_test/missing_tool.jsonl --method lookback_lens --lookback_classifier models/lookback_lens --device cuda


Loaded 486 examples from final_dataset_test/missing_tool.jsonl
Building predictor: method=lookback_lens, lettuce_model=KRLabsOrg/lettucedect-base-modernbert-en-v1, lookback_classifier=models/lookback_lens, device=cuda
Loading weights: 100% 201/201 [00:01<00:00, 113.77it/s, Materializing param=model.norm.weight]
Predictor is ready. Starting evaluation...
Evaluating lookback_lens: 100% 486/486 [03:37<00:00,  2.24example/s]
Method=lookback_lens  TP=486 FP=621 FN=1 P=0.4390 R=0.9979 F1=0.6098
Saved evaluation results to models/lookback_lens/eval_results.json


In [ ]:
!python src/eval_baselines.py --dataset final_dataset_test/clean.jsonl --method lookback_lens --lookback_classifier models/lookback_lens --device cuda

Loaded 486 examples from final_dataset_test/clean.jsonl
Building predictor: method=lookback_lens, lettuce_model=KRLabsOrg/lettucedect-base-modernbert-en-v1, lookback_classifier=models/lookback_lens, device=cuda
Loading weights: 100% 201/201 [00:01<00:00, 116.40it/s, Materializing param=model.norm.weight]
Predictor is ready. Starting evaluation...
Evaluating lookback_lens: 100% 486/486 [03:16<00:00,  2.47example/s]
Method=lookback_lens  TP=0 FP=834 FN=0 P=0.0000 R=0.0000 F1=0.0000
Saved evaluation results to models/lookback_lens/eval_results.json


### 3.4.4 Ensamble

In [ ]:
# Ensamble
!python src/run_voting_ensemble_experiment.py --dataset_dir final_dataset --methods tool_overlap,lettucedetect,lookback_lens --seed 42 --test_ratio 0.2 --tune_ratio 0.25 --device cuda \
--lettuce_device cuda --lookback_device cpu --lookback_model sshleifer/tiny-gpt2 --lookback_max_length 256 --prediction_splits tune,test --output_dir outputs\ensemble_voting\notebook_full

### 3.4.5 Fine-Tuned Span Model

In [ ]:
!python src/finetuned_span_model.py train \
  --dataset_dir final_dataset_train \
  --output_dir models/experiment_margo \
  --device cuda \
  --fp16 \
  --gradient_checkpointing

Loaded 7780 examples
Train examples: 6670
Validation examples: 1110
config.json: 1.27kB [00:00, 4.89MB/s]
tokenizer_config.json: 20.8kB [00:00, 59.6MB/s]
tokenizer.json: 3.58MB [00:00, 132MB/s]
special_tokens_map.json: 100% 694/694 [00:00<00:00, 4.87MB/s]
Tokenizing train: 100% 6670/6670 [00:17<00:00, 387.81example/s]
Tokenizing validation: 100% 1110/1110 [00:03<00:00, 299.03example/s]
model.safetensors: 100% 598M/598M [00:03<00:00, 165MB/s]
Loading weights: 100% 138/138 [00:00<00:00, 1362.58it/s, Materializing param=model.layers.21.mlp_norm.weight]
Class weights: supported=0.5373, hallucination=7.1985
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
  0% 0/1668 [00:00<?, ?it/s]W0526 16:4

In [ ]:
!python src/finetuned_span_model.py evaluate \
  --dataset final_dataset_test \
  --model_dir models/experiment_margo \
  --device cuda \
  --metrics_out outputs/experiment_margo_metrics.json

Loading weights: 100% 138/138 [00:00<00:00, 2133.57it/s, Materializing param=model.layers.21.mlp_norm.weight]
Predicting: 100% 1944/1944 [01:23<00:00, 23.20example/s]
Overall: TP=1376 FP=113 FN=90 P=0.9241 R=0.9386 F1=0.9313 N=1944
clean: TP=0 FP=54 FN=0 P=0.0000 R=0.0000 F1=0.0000 N=486
contradiction: TP=403 FP=42 FN=86 P=0.9056 R=0.8241 F1=0.8630 N=486
missing_tool: TP=484 FP=7 FN=2 P=0.9857 R=0.9959 F1=0.9908 N=486
overgeneration: TP=489 FP=10 FN=2 P=0.9800 R=0.9959 F1=0.9879 N=486
